<a href="https://colab.research.google.com/github/diseasemodeling/MIE_525_625/blob/main/Lectures/4_MLP_heads/4b_LLM_MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TEXT Classfiication: classify article by newsgroup type.
##  Data: X=news article and Y=newsgroup (20 newsgroups) for supervised learning;
GOAL: classiify a news article by newsgroup
MODEL:
* Use pretrained encoder only LLM from hugging face to convert text to numbers.
* Apply MLP for classification.



Dataset Source:
http://qwone.com/~jason/20Newsgroups/   
You can read about the dataset and full list of groups in the source.

## SIMPLIFIED DATA:
* To use a simpe example, here we pick 2 categories for a 2-class classfication problem.
* ['sci.space', 'comp.graphics']


Using sckitlearn datasets, as it is cleaned up.  
https://scikit-learn.org/0.19/datasets/twenty_newsgroups.html

NOte: used AI to help generate this code through prompting

In [ ]:
#!pip install -q torch transformers datasets shap scikit-learn pandas numpy

In [ ]:
import numpy as np
import torch
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score
from transformers import AutoTokenizer, AutoModel

In [ ]:
# =====================================================================
# 1. FETCH DATASET FROM SCIKIT-LEARN
# =====================================================================
print("📥 Fetching the official 20 Newsgroups text dataset...")
# We pick two highly contrasting categories for a clean binary classification task
categories = ['sci.space', 'comp.graphics']
newsgroups = fetch_20newsgroups(subset='all', categories=categories, remove=('headers', 'footers', 'quotes'))

# Take a quick sample of 200 items for rapid CPU execution
texts = newsgroups.data[:200]
y = newsgroups.target[:200]

📥 Fetching the official 20 Newsgroups text dataset...


In [ ]:
#print summary and sample
print(f"Dataset successfully loaded. Samples: {len(texts)}")
print(f"Target Labels: 0 = {newsgroups.target_names[0]}, 1 = {newsgroups.target_names[1]}")
print(f"Sample Post text:\n\"{texts[0][:150]}...\"\n")

Dataset successfully loaded. Samples: 2
Target Labels: 0 = comp.graphics, 1 = sci.space
Sample Post text:
"
Actually, Hiten wasn't originally intended to go into lunar orbit at all,
so it indeed didn't have much fuel on hand.  The lunar-orbit mission was
an..."



In [ ]:
# Print 5 samples to get a sense for how data looks by printing first 150 characters of each sample, and its labe;
for i in range(5):
    print(f"--- Sample #{i+1} ---")
    print(f"Label (y): {y[i]} ({newsgroups.target_names[y[i]]})")
    print(f"Text snippet:\n{texts[i][:150]}...") # limit text to 150 characters for clean console reading
    print("-" * 40, "\n")


--- Sample #1 ---
Label (y): 1 (sci.space)
Text snippet:

Actually, Hiten wasn't originally intended to go into lunar orbit at all,
so it indeed didn't have much fuel on hand.  The lunar-orbit mission was
an...
---------------------------------------- 

--- Sample #2 ---
Label (y): 1 (sci.space)
Text snippet:
Archive-name: space/launchers
Last-modified: $Date: 93/04/01 14:39:11 $

ORBITAL AND PLANETARY LAUNCH SERVICES

The following data comes from _Interna...
---------------------------------------- 

--- Sample #3 ---
Label (y): 1 (sci.space)
Text snippet:

Neil Gehrels is Prof. Tom Gehrels son.  Tom Gehrels was the discoverer
of P/Gehrels 3 (as well as about 4 other comets - the latest of which
does not...
---------------------------------------- 

--- Sample #4 ---
Label (y): 1 (sci.space)
Text snippet:
I have the "osculating elements at perigee" of an orbit, which I need
to convert to something useful, preferably distance from the earth
in evenly spa...
---------------------------

In [ ]:
# =====================================================================
# 2. FEATURE EXTRACTION USING AN LLM (DistilBERT Embeddings)
# =====================================================================
print("🤖 Extracting LLM embeddings...")
model_name = "distilbert-base-uncased" # while ChatGPT is decoder only LLM, this one is an encoder only LLM. It encodes text to numbers.
tokenizer = AutoTokenizer.from_pretrained(model_name) # convert text to numbers
model = AutoModel.from_pretrained(model_name)
#print model size
print(f"Model size: {sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters") #It is a small model compared to modern day LLM. And "small" here is in order of millions. LLM have billion to trillion parametrs (model weights)

🤖 Extracting LLM embeddings...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model size: 66.36M parameters


In [ ]:
def extract_llm_embeddings(text_list):
    embeddings = []
    model.eval() # notice use of pretrained model, we directly go to model eval.
    with torch.no_grad():
        for text in text_list:
            # Handle empty posts gracefully
            if not text.strip():
                text = "empty post"
            inputs = tokenizer(text, padding=True, truncation=True, max_length=128, return_tensors="pt")
            outputs = model(**inputs) # forward pass of LLM "distilbert-base-uncased"
            cls_embedding = outputs.last_hidden_state[0, 0, :].numpy()
            embeddings.append(cls_embedding)
    return np.array(embeddings)

X = extract_llm_embeddings(texts) # Outputs from LLM are used as inputs in MLP
print(f"Embeddings generated with shape: {X.shape}\n")

In [ ]:
# print sample text snippet and corresponding X,
print(f"Sample Post text:\n\"{texts[0][:150]}...\"\n")
#print(f"Sample X:\n{X[0]}\n")
#print shape of sample text and shape of X; notice each sample has varying text length but they are all converted to same length in X
sample = 4
print(f"Text length (characters): {len(texts[sample])}")
print(f"X shape for each sample: {X[sample].shape}")
print(f"X shape: {X.shape}")

Sample Post text:
"
Actually, Hiten wasn't originally intended to go into lunar orbit at all,
so it indeed didn't have much fuel on hand.  The lunar-orbit mission was
an..."

Text length (characters): 2078
X shape for each sample: (768,)
X shape: (200, 768)


In [ ]:
# =====================================================================
# 3. SPLIT DATA & TRAIN A MULTI-LAYER PERCEPTRON (MLP)
# =====================================================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("🧠 Training the Multi-Layer Perceptron (MLP)...")
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', max_iter=300, random_state=42) # UNlike Pytorch sk-learn has a one-line code for MLP.
mlp.fit(X_train, y_train)

In [ ]:
# =====================================================================
# 4. EVALUATION
# =====================================================================
y_pred = mlp.predict(X_test)
print(f"🎯 Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(classification_report(y_test, y_pred, target_names=newsgroups.target_names))

🎯 Accuracy: 92.50%
               precision    recall  f1-score   support

comp.graphics       0.90      0.95      0.93        20
    sci.space       0.95      0.90      0.92        20

     accuracy                           0.93        40
    macro avg       0.93      0.93      0.92        40
 weighted avg       0.93      0.93      0.92        40

